# Five-fold ConvNeXt ensemble: saved weights to submission

No training or threshold tuning here. Read `ensemble_config.json` created after all five
folds in `cnn_2_cv.ipynb`, restore one model at a time, average sigmoid probabilities
with equal weights, then apply the saved OOF thresholds. All models remain ConvNeXt-Tiny.

Attach the competition data and the complete `convnext_cv` output (all five `best.pt`
files and `ensemble_config.json`). Set `CV_DIR` explicitly if it is mounted under Input.
`last.pt` and W&B credentials are not needed for inference.

Outputs in `/kaggle/working/convnext_cv_ensemble`:
* **submission_cv_tta.csv** — five-model ensemble plus the original/flip TTA.
* submission_cv.csv — five-model ensemble without TTA, for comparison.

Per-fold probabilities are cached with checkpoint hashes and ordered image IDs to avoid
repeating completed GPU inference after a same-session interruption. An invalid cache
is recomputed. Download/cache this directory separately if you want to resume inference
after a new Kaggle session. No Kaggle submission is uploaded automatically.

In [1]:
# Optional install cell; restart the kernel afterwards.
# %pip install -q timm "albumentations>=2,<3"

In [2]:
import json
import os
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, FileLink
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
import timm

CV_DIR = Path('/kaggle/working/convnext_cv')
DATA_DIR = Path('/kaggle/input/competitions/cats-and-dogs')
OUTPUT_DIR = Path('/kaggle/working/convnext_cv_ensemble')
BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def file_digest(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

if not (CV_DIR / 'ensemble_config.json').is_file():
    candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
        if 'ensemble_config.json' in files and all((Path(directory) / f'fold_{f}' / 'best.pt').is_file() for f in range(5)):
            candidates.append(Path(directory))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Finish all five folds and attach their complete Output, or set CV_DIR. Candidates: {candidates}')
    CV_DIR = candidates[0]
ensemble_config = json.loads((CV_DIR / 'ensemble_config.json').read_text())
assert ensemble_config['model_name'] == 'convnext_tiny.fb_in1k'
assert ensemble_config['folds'] == list(range(5)), 'A full five-fold ensemble is required.'
assert ensemble_config['aggregation'] == 'equal_mean_sigmoid_probabilities'
LABEL_COLS = ensemble_config['label_cols']
assert LABEL_COLS == ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
cutoffs = {key: np.asarray(ensemble_config['thresholds'][key], dtype=np.float64) for key in ['original', 'tta']}
for values in cutoffs.values():
    assert values.shape == (len(LABEL_COLS),) and np.isfinite(values).all()
    assert ((values > 0) & (values < 1)).all()
preprocessing = ensemble_config['preprocessing']
assert preprocessing['name'] == 'albumentations_resize_v1'
import albumentations as A
from albumentations.pytorch import ToTensorV2
eval_transform = A.Compose([
    A.Resize(preprocessing['image_size'], preprocessing['image_size'], interpolation=preprocessing['interpolation']),
    A.Normalize(mean=preprocessing['mean'], std=preprocessing['std']), ToTensorV2(),
])
print('device:', DEVICE, '| CV outputs:', CV_DIR)

device: cuda | CV outputs: /kaggle/working/convnext_cv


## 1. Test rows

Keep exact test IDs and check the official sample schema. No training labels are read.

In [3]:
if not (DATA_DIR / 'test.csv').is_file():
    candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
        root = Path(directory)
        if {'train.csv', 'test.csv', 'sample_submission.csv'}.issubset(files):
            columns = pd.read_csv(root / 'train.csv', nrows=0).columns
            if {'image_id', *LABEL_COLS}.issubset(columns) and (root / 'train').is_dir() and (root / 'test').is_dir():
                candidates.append(root)
    if len(candidates) != 1:
        raise FileNotFoundError(f'Attach the competition data or set DATA_DIR. Candidates: {candidates}')
    DATA_DIR = candidates[0]


test = pd.read_csv(DATA_DIR / 'test.csv', dtype={'image_id': str})
assert len(test) > 0 and test.image_id.notna().all() and test.image_id.is_unique
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'image_id': str})
assert set(sample.columns) == {'image_id', *LABEL_COLS}
assert sample.image_id.notna().all() and sample.image_id.is_unique
assert set(sample.image_id) == set(test.image_id)

class TestImages(Dataset):
    def __len__(self):
        return len(test)
    def __getitem__(self, index):
        with Image.open(DATA_DIR / 'test' / f'{test.iloc[index].image_id}.jpg') as image:
            return eval_transform(image=np.array(image.convert('RGB')))['image']

test_loader = DataLoader(TestImages(), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Test images:', len(test))

Test images: 18673


In [4]:
@torch.inference_mode()
def predict_pair(loader, description):
    model.eval()
    original_batches, tta_batches = [], []
    for images in tqdm(loader, desc=description):
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        original = logits.float().sigmoid()
        del logits
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            flipped_logits = model(torch.flip(images, dims=[3]))
        averaged = (original + flipped_logits.float().sigmoid()) * 0.5
        original_batches.append(original.cpu().numpy())
        tta_batches.append(averaged.cpu().numpy())
    original = np.concatenate(original_batches)
    tta = np.concatenate(tta_batches)
    for probabilities in (original, tta):
        assert probabilities.shape == (len(loader.dataset), len(LABEL_COLS))
        assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    return original, tta

## 2. Average probabilities across five checkpoints

Use one model on the GPU at a time. Verify each checkpoint against the OOF-calibrated ensemble manifest. No averaging of class IDs or independently thresholded predictions.

In [5]:
mean_original = np.zeros((len(test), len(LABEL_COLS)), dtype=np.float64)
mean_tta = np.zeros_like(mean_original)
test_ids = test.image_id.to_numpy(dtype=str)
for fold in ensemble_config['folds']:
    checkpoint_path = CV_DIR / f'fold_{fold}' / 'best.pt'
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f'Missing fold {fold} checkpoint: {checkpoint_path}')
    checkpoint_hash = file_digest(checkpoint_path)
    assert checkpoint_hash == ensemble_config['checkpoint_hashes'][str(fold)], 'Checkpoint changed after OOF calibration.'
    cache_path = OUTPUT_DIR / f'fold_{fold}_test.npz'
    cache_ok = False
    if cache_path.is_file():
        try:
            with np.load(cache_path, allow_pickle=False) as cache:
                original, averaged = cache['original'], cache['tta']
                cache_ok = (str(cache['checkpoint_hash'].item()) == checkpoint_hash
                            and np.array_equal(cache['image_ids'], test_ids)
                            and np.array_equal(cache['label_cols'], np.array(LABEL_COLS)))
                for probabilities in (original, averaged):
                    cache_ok = cache_ok and probabilities.shape == mean_original.shape and np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
        except (OSError, ValueError, KeyError):
            cache_ok = False
    if cache_ok:
        print(f'Fold {fold}: using verified cached probabilities.')
    else:
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        cfg = checkpoint['config']
        assert cfg['cv_fingerprint'] == ensemble_config['cv_fingerprint'] and cfg['fold'] == fold
        assert cfg['model_name'] == ensemble_config['model_name'] and cfg['label_cols'] == LABEL_COLS
        assert cfg['preprocessing'] == preprocessing
        model = timm.create_model(cfg['model_name'], pretrained=False, num_classes=len(LABEL_COLS)).to(DEVICE)
        model.load_state_dict(checkpoint['model'], strict=True)
        del checkpoint
        original, averaged = predict_pair(test_loader, f'fold {fold}: test original + flip')
        temporary = cache_path.with_suffix('.tmp.npz')
        np.savez_compressed(temporary, original=original, tta=averaged, image_ids=test_ids,
                            label_cols=np.array(LABEL_COLS), checkpoint_hash=np.array(checkpoint_hash))
        temporary.replace(cache_path)
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    mean_original += original.astype(np.float64) / len(ensemble_config['folds'])
    mean_tta += averaged.astype(np.float64) / len(ensemble_config['folds'])
for probabilities in (mean_original, mean_tta):
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()

fold 4: test original + flip: 100%|██████████| 584/584 [01:59<00:00,  4.87it/s]


## 3. Export submissions

Submit **submission_cv_tta.csv** first to compare with 0.93720. `submission_cv.csv` is the ensemble without TTA. Both use their matching OOF thresholds and preserve the official sample row order.

In [6]:
for method, probabilities, filename in [('original', mean_original, 'submission_cv.csv'),
                                         ('tta', mean_tta, 'submission_cv_tta.csv')]:
    probability_frame = pd.DataFrame(probabilities, columns=LABEL_COLS)
    probability_frame.insert(0, 'image_id', test.image_id.to_numpy())
    probability_frame.to_csv(OUTPUT_DIR / f'test_probabilities_{method}.csv', index=False)
    submission = probability_frame.copy()
    submission[LABEL_COLS] = (probabilities >= cutoffs[method]).astype(np.int32)
    submission = sample[['image_id']].merge(submission, on='image_id', how='left', validate='one_to_one')
    submission = submission[['image_id', *LABEL_COLS]]
    assert len(submission) == len(test) and not submission.isna().any().any()
    assert submission[LABEL_COLS].isin([0, 1]).all().all()
    path = OUTPUT_DIR / filename
    submission.to_csv(path, index=False)
    print('Saved:', path, '| rows:', len(submission))
    display(FileLink(str(path)))
(OUTPUT_DIR / 'ensemble_config.json').write_text(json.dumps(ensemble_config, indent=2))
display(submission.head())

Saved: /kaggle/working/convnext_cv_ensemble/submission_cv.csv | rows: 18673


/kaggle/working/convnext_cv_ensemble/submission_cv.csv

Saved: /kaggle/working/convnext_cv_ensemble/submission_cv_tta.csv | rows: 18673


/kaggle/working/convnext_cv_ensemble/submission_cv_tta.csv

,image_id,Cat,Dog,Moris,Motya,Biatrix
0,12c79b0a4bd444ca7e5f730b86824b9279d3cd24,0,0,0,0,0
1,dd9440332972c9c5cecb5bf3cfb3329d6b2e422c,0,1,0,0,0
2,ac89f642faef21c92d622d882943e9065600354a,0,0,0,0,0
3,7245e7046ae54da95dbddc24ae814b3c61cc6237,1,0,0,0,0
4,d2b76179615de57b54cb00d094fba50810d96127,1,0,0,0,0
